# 05 · LightGBM global model

Drives `m5.models.lgbm.build_lgbm_forecaster` and `m5.cv.lgbm_cv` — the 
same path `make cv-lgbm` and `make forecast-lgbm` use. Notebook outputs 
match CLI outputs byte-for-byte.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

from m5.config import SETTINGS, set_global_seed
from m5.cv import lgbm_cv
from m5.evaluation import compute_components, make_submission, wrmsse_for_models
from m5.models.lgbm import (
    DEFAULT_LAGS,
    DEFAULT_ROLLS,
    build_lgbm_forecaster,
    fit_predict_lgbm,
    lgbm_params,
)
from m5.plots import configure_style

configure_style()
set_global_seed()

42

## Inspect the canonical hyperparameters

Tweedie objective, `deterministic=True`, `force_row_wise=True`, 
`seed=SETTINGS.seed`, lags=(7, 14, 28), rolling means over (7, 28) lagged by 1, 
`Differences([1])`, date_features=`[dayofweek, day, week, month, year]`. 
Two notebook runs produce identical numbers and match the CLI.

In [3]:
fcst = build_lgbm_forecaster()
params = fcst.models['LGBM'].get_params()
{k: params[k] for k in ('objective', 'tweedie_variance_power', 'learning_rate',
                         'num_leaves', 'n_estimators', 'seed', 'deterministic')}

{'objective': 'tweedie',
 'tweedie_variance_power': 1.1,
 'learning_rate': 0.05,
 'num_leaves': 128,
 'n_estimators': 1500,
 'seed': 42,
 'deterministic': True}

### To experiment with overrides

Take the canonical defaults and override only the knob you want to vary; 
everything else stays pinned, so the comparison is clean.

In [4]:
# import lightgbm as lgb
# from mlforecast import MLForecast
# from mlforecast.lag_transforms import RollingMean
#
# custom = {**lgbm_params(), 'num_leaves': 256, 'n_estimators': 2000}
# experimental = MLForecast(
#     models={'LGBM': lgb.LGBMRegressor(**custom)}, freq='D',
#     lags=list(DEFAULT_LAGS),
#     lag_transforms={1: [RollingMean(window_size=w) for w in DEFAULT_ROLLS]},
#     date_features=['dayofweek', 'day', 'week', 'month', 'year'],
# )

## Cross-validation

In [5]:
df = pd.read_parquet(SETTINGS.processed_dir / 'long.parquet')

cv = lgbm_cv(df, h=SETTINGS.horizon, n_windows=SETTINGS.n_windows)
cv.to_parquet(SETTINGS.artifacts_dir / 'cv_lgbm.parquet', index=False)
cv.tail()

12:56:33 | INFO    | m5.cv:lgbm_cv:68 - lgbm_cv: h=28 n_windows=3 step=28
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/mlforecast/core.py:683: UserWarning: The following series were dropped completely due to the transformations and features: ['FOODS_3_595_CA_1', 'HOUSEHOLD_1_020_WI_2', 'HOUSEHOLD_1_278_CA_3'].
These series won't show up if you use `MLForecast.forecast_fitted_values()`.
You can set `dropna=False` or use transformations that require less samples to mitigate this
  warnings.warn(
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/mlforecast/core.py:1030: UserWarning: Found null values in lag14, lag28, rolling_mean_lag1_window_size28.
  warnings.warn(f'Found null values in {", ".join(cols_with_nulls)}.')
/home/ricka/Git/GitHub/M5/.venv/lib/python3.12/site-packages/mlforecast/core.py:1030: UserWarning: Found null values in lag14, lag28, rolling_mean_lag1_window_size28.
  warnings.warn(f'Found null values in {", ".join(cols_with_nulls)}.')
/home/ric

,unique_id,ds,cutoff,y,LGBM
419995,HOUSEHOLD_2_516_WI_3,2016-05-18,2016-04-24,0.0,0.049850
419996,HOUSEHOLD_2_516_WI_3,2016-05-19,2016-04-24,0.0,0.056699
419997,HOUSEHOLD_2_516_WI_3,2016-05-20,2016-04-24,0.0,0.066731
419998,HOUSEHOLD_2_516_WI_3,2016-05-21,2016-04-24,0.0,0.092802
419999,HOUSEHOLD_2_516_WI_3,2016-05-22,2016-04-24,0.0,0.087257


In [6]:
components = compute_components(df[df['ds'] < cv['ds'].min()])
wrmsse_for_models(cv[['unique_id', 'ds', 'y']], cv, components)

LGBM    0.818543
Name: wrmsse, dtype: float64

## Train on full data + emit forecast

In [7]:
forecast = fit_predict_lgbm(df, horizon=SETTINGS.horizon)
forecast.to_parquet(SETTINGS.forecasts_dir / 'forecast_lgbm.parquet', index=False)
forecast.head()

13:01:46 | INFO    | m5.models.lgbm:fit_lgbm:110 - fit_lgbm: features → 5,000 series × 200 rows each
13:01:47 | INFO    | m5.models.lgbm:fit_lgbm:122 - fit_lgbm: fitting LightGBM (dynamic=0 cols)
13:03:11 | INFO    | m5.models.lgbm:fit_lgbm:124 - fit_lgbm: fit done in 84.8s
13:03:11 | INFO    | m5.models.lgbm:fit_predict_lgbm:143 - fit_predict_lgbm: predicting (h=28)
13:03:15 | INFO    | m5.models.lgbm:fit_predict_lgbm:145 - fit_predict_lgbm: total 88.8s, 140,000 forecast rows


,unique_id,ds,LGBM
0,FOODS_1_001_CA_1,2016-05-23,0.872862
1,FOODS_1_001_CA_1,2016-05-24,0.729882
2,FOODS_1_001_CA_1,2016-05-25,0.727841
3,FOODS_1_001_CA_1,2016-05-26,0.652990
4,FOODS_1_001_CA_1,2016-05-27,0.615181


## Optional — Kaggle-format submission

In [8]:
submission = make_submission(forecast, h=SETTINGS.horizon, value_col='LGBM', layout='kaggle')
submission.head()

,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
unique_id,,,,,,,,,,,,,,,,,,,,,
FOODS_1_001_CA_1,0.872862,0.729882,0.727841,0.652990,0.615181,0.721226,0.677858,0.586289,0.689421,0.766892,...,0.787121,0.761729,0.819567,0.817735,0.813313,0.791032,0.581294,0.613905,0.683881,0.661966
FOODS_1_001_TX_2,0.406283,0.376124,0.445985,0.363653,0.350684,0.555021,0.582400,0.352926,0.406531,0.529852,...,0.512184,0.592303,0.651551,0.579811,0.687325,0.589910,0.541825,0.628737,0.676728,0.576822
FOODS_1_003_CA_4,0.292929,0.307406,0.273338,0.276819,0.223158,0.264673,0.211856,0.304806,0.354665,0.218054,...,0.321783,0.244822,0.198099,0.279191,0.307962,0.288613,0.330221,0.313469,0.246798,0.221766
FOODS_1_003_TX_3,0.112109,0.178937,0.153896,0.159842,0.164685,0.181281,0.185920,0.189382,0.264141,0.137493,...,0.141588,0.154458,0.130575,0.156712,0.179862,0.167607,0.169695,0.144052,0.153444,0.140352
FOODS_1_003_WI_1,1.021902,1.524442,1.201848,1.252153,1.518316,1.553044,1.224767,1.104504,1.287192,0.890201,...,1.660021,1.555513,1.218421,1.018449,1.146295,1.205752,1.422426,1.556457,1.852897,1.170467
